# Pipeline d'Inférence d'Images avec TensorFlow
Ce notebook présente un pipeline complet d'inférence d'images à l'aide de TensorFlow et Keras. Il inclut la classification, le débruitage et la génération de légendes.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt
import tensorflow.keras.backend as K

## Paramètres globaux

In [7]:
IMG_SIZE = (256, 256, 3)
BATCH_SIZE = 32
DATA_DIR = "./Validation_photos"  
CLASS_NAMES = ['Photo', 'Sketch', 'Painting', 'Text', 'Schematics']
LABEL_TO_KEEP = 'Photo'

## Chargement des modèles entraînés

In [8]:
class Autoencoder(tf.keras.Model):
    def __init__(self, encoder, decoder, latent_dim, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.latent_dim = latent_dim

    def call(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

    def get_config(self):
        config = super().get_config()
        config.update({
            "encoder": tf.keras.utils.serialize_keras_object(self.encoder),
            "decoder": tf.keras.utils.serialize_keras_object(self.decoder),
            "latent_dim": self.latent_dim
        })
        return config

    @classmethod
    def from_config(cls, config):
        print("DEBUG CONFIG:", config)  # Ajoute ceci
        encoder = tf.keras.utils.deserialize_keras_object(config.pop("encoder"))
        decoder = tf.keras.utils.deserialize_keras_object(config.pop("decoder"))
        return cls(encoder=encoder, decoder=decoder, **config)
    
@tf.keras.utils.register_keras_serializable()
def combined_mse_ssim_loss(y_true, y_pred):
    # MSE pour la précision pixel par pixel
    mse_loss = K.mean(K.square(y_true - y_pred))

    # SSIM pour la préservation de la structure
    ssim = 1 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))

    # Combinaison pondérée
    alpha = 0.84  # Ajustez ce coefficient selon vos besoins
    return (1 - alpha) * mse_loss + alpha * ssim

In [9]:
@tf.keras.utils.register_keras_serializable()
def weighted_loss(y_true, y_pred):
    weights = tf.gather(class_weight_tensor, tf.cast(y_true, tf.int32))
    unweighted_loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    return unweighted_loss * weights

In [10]:
classification_model = load_model("./L1_model.keras")
autoencoder = load_model("./L2_model.keras", custom_objects={'Autoencoder': Autoencoder})
# caption_model = load_model("./L3_model.keras")  # Doit inclure tokenizer/décodage

DEBUG CONFIG: {'name': 'autoencoder', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None, 'shared_object_id': 13951487664}, 'encoder': {'module': 'keras', 'class_name': 'Sequential', 'config': {'name': 'sequential', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None, 'shared_object_id': 13951487664}, 'layers': [{'module': 'keras.layers', 'class_name': 'InputLayer', 'config': {'batch_shape': [None, 256, 256, 3], 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_layer'}, 'registered_name': None}, {'module': 'keras.layers', 'class_name': 'Conv2D', 'config': {'name': 'conv2d', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None, 'shared_object_id': 13951487664}, 'filters': 32, 'kernel_size': [3, 3], 'strides': [1, 1], 'padding': 'sam

## Fonctions de traitement du pipeline

In [11]:
def pre_processing_dataset(data_dir, img_size=IMG_SIZE, batch_size=BATCH_SIZE):
    # Chargement des images
    dataset = tf.keras.preprocessing.image_dataset_from_directory(
        data_dir,
        labels=None,
        image_size=img_size,
        batch_size=batch_size,
        shuffle=False
    )
    
    dataset = dataset.map(lambda x: x / 255.0) # Normalisation des images entre 0 et 1

    return dataset

In [12]:
# Filtrage des images reconnues comme LABEL_TO_KEEP
def filter_only_photos(batch):
    # Prédire les classes
    predictions = classification_model.predict(batch)
    # Obtenir les indices des classes
    predicted_classes = tf.argmax(predictions, axis=1)
    # Filtrer les images qui sont de type LABEL_TO_KEEP
    filtered_images = tf.boolean_mask(batch, predicted_classes == CLASS_NAMES.index(LABEL_TO_KEEP))
    return filtered_images    

In [13]:
# Débruitage des images via autoencodeur
def denoise_images(batch):
    return autoencoder(batch, training=False)

In [14]:
# Génération de légendes
def generate_captions(batch):
    batch_size = tf.shape(batch)[0]
    dummy_caption = tf.constant("Légende générique", dtype=tf.string) # Remplacer par le modèle de génération de légende
    captions = tf.repeat(dummy_caption, batch_size)
    return batch, captions

## Construction du pipeline rejouable avec `tf.data.Dataset`

In [15]:
# Build pipeline object
class Pipeline:
    def __init__(self, data_dir, img_size=IMG_SIZE, batch_size=BATCH_SIZE):
        self.data_dir = data_dir
        self.img_size = img_size
        self.batch_size = batch_size

        self.dataset = pre_processing_dataset(self.data_dir, self.img_size, self.batch_size)

    def process(self):
        print("DEBUG DATASET SHAPE:", self.dataset.element_spec)

        # Filtrage des images
        filtered_dataset = self.dataset.map(lambda x: filter_only_photos(x))
        print("DEBUG FILTERED DATASET SHAPE:", filtered_dataset.element_spec)

        # Dénombrement des images
        #denoised_dataset = filtered_dataset.map(lambda x: denoise_images(x))
        #print("DEBUG DENOISED DATASET SHAPE:", denoised_dataset.element_spec)

        # Génération de légendes
        #final_dataset = denoised_dataset.map(lambda x: generate_captions(x))
        #print("DEBUG FINAL DATASET SHAPE:", final_dataset.element_spec)

        # return final_dataset
        return filtered_dataset

## Utilisation du pipeline et affichage des résultats

In [25]:
pipeline = Pipeline(DATA_DIR, (IMG_SIZE[0], IMG_SIZE[1]), BATCH_SIZE)

processed_dataset = pipeline.process()

# Affichage de la forme du dataset
for images, captions in processed_dataset.take(1):
    print("DEBUG: Processed dataset shape:", images.shape, captions.shape)

# Visualisation d'un échantillon d'images et de leurs légendes
for images, captions in pipeline.take(1):
    num_samples = min(5, images.shape[0])
    plt.figure(figsize=(15, 8))
    for i in range(num_samples):
        ax = plt.subplot(1, num_samples, i + 1)
        plt.imshow(images[i].numpy())
        plt.title(captions[i].numpy().decode())
        plt.axis("off")
    plt.show()

Found 148 files.
DEBUG DATASET SHAPE: TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name=None)


TypeError: in user code:

    File "/var/folders/_6/67rhg3t936xc3_p_pxrngkn40000gs/T/ipykernel_3603/1648705587.py", line 14, in None  *
        lambda x: filter_only_photos(x)
    File "/var/folders/_6/67rhg3t936xc3_p_pxrngkn40000gs/T/ipykernel_3603/1338562429.py", line 4, in filter_only_photos  *
        predictions = classification_model.predict(batch)
    File "/Users/laurine_/Documents/CESI/projet-datascience-a5/.venv/lib/python3.10/site-packages/keras/src/utils/traceback_utils.py", line 122, in error_handler  **
        raise e.with_traceback(filtered_tb) from None
    File "/Users/laurine_/Documents/CESI/projet-datascience-a5/.venv/lib/python3.10/site-packages/keras/src/trainers/data_adapters/data_adapter_utils.py", line 104, in <genexpr>
        num_samples = set(int(i.shape[0]) for i in tree.flatten(data))

    TypeError: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'
